# Deep learning on images

## Load modules from repo

In [5]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [6]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [7]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features
from src.preprocessing.pipelines.deep_learning import load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-06 15:55:14.524928: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 15:55:14.566477: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 15:55:15.504293: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1759758916.203004   14413 gpu_device.cc:2020] Created device /job:localhost/rep

In [8]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_y_train=y_train

In [9]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [ ]:
modality = 'image'
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = False  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

# augment=True  # Augment data for training
augment=False

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False

## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [13]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

In [14]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

In [15]:
y_train.value_counts().describe()

count      27.000000
mean     2516.000000
std      1682.489662
min       611.000000
25%      1241.000000
50%      2137.000000
75%      3813.500000
max      8167.000000
Name: count, dtype: float64

In [16]:
print(X_train.shape)

(67932, 31)


In [ ]:
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [18]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [21]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Load or create model

In [ ]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    arch_version = last_experiment.get('arch_version', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
        _, base_model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    arch_version = last_experiment.get('arch_version', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model, base_model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)


Reprise depuis le meilleur modèle de l'expérience : artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-01_val_accuracy-0.6365_f1-0.6253.keras


In [ ]:
if not load_model:
    arch_version = int(input(f"arch_version ? (last: {arch_version})"))

In [ ]:
arch_version

9

### Summary

In [ ]:
model.summary()

## Callbacks

### ModelCheckpoint

In [ ]:
# # Pick an available filename to save a model.
# arch_version=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{arch_version}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     arch_version+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{arch_version}.h5')
# new_location_for_saving_model


In [28]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [29]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [30]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_accuracy', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='max',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [32]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [33]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [ ]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder / f"{timestamp}-{modality}",
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [35]:
import math
# typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793
typical_minutes_per_epoch=8.6 * X_train.shape[0] / 67932

In [36]:
max_epochs=10

# Calculate expected duration
available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
available_minutes

86

In [37]:
# # Pick max_epochs based on your available time
# available_minutes=60

# max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
# max_epochs

### compilation and callbacks

In [38]:
learning_rate=0.001

In [39]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [40]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [41]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=86, max_epochs=10, champion_path='artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-01_val_accuracy-0.6365_f1-0.6253.keras' ?

In [ ]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=total_epochs_trained, callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 2/12


2025-10-06 15:03:31.447322: I external/local_xla/xla/service/service.cc:163] XLA service 0x740ca8012610 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-06 15:03:31.447374: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-06 15:03:31.741653: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-06 15:03:33.124077: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-06 15:03:33.994811: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-06 15:03:33.

2122/2123 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6916 - loss: 1.4149

2025-10-06 15:08:18.987524: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-06 15:08:28.352306: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 15:08:28.451780: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 15:08:29.391254: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

2123/2123 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.6916 - loss: 1.4148

2025-10-06 15:09:49.660865: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-06 15:09:57.921120: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 15:09:58.019515: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 15:09:58.906629: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

2123/2123 ━━━━━━━━━━━━━━━━━━━━ 402s 175ms/step - accuracy: 0.7273 - loss: 1.3186 - val_accuracy: 0.6062 - val_loss: 1.7152 - learning_rate: 0.0010
Epoch 3/12
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 331s 156ms/step - accuracy: 0.8399 - loss: 1.0293 - val_accuracy: 0.5281 - val_loss: 2.0310 - learning_rate: 0.0010
Epoch 4/12
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.8884 - loss: 0.8822
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 330s 155ms/step - accuracy: 0.9165 - loss: 0.7879 - val_accuracy: 0.5081 - val_loss: 2.0886 - learning_rate: 0.0010
Epoch 5/12
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 328s 154ms/step - accuracy: 0.9559 - loss: 0.6293 - val_accuracy: 0.5175 - val_loss: 2.0067 - learning_rate: 2.0000e-04
Epoch 6/12
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.9544 - loss: 0.6312
Epoch 6: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.
2123/2123 ━━━━━━━━━━━━━━━━━━━━ 329s 155ms/step - acc

'total_minutes=34.12972358862559'

## Evaluation

In [43]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensorboard_logs'


In [44]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 7,
 'minutes_per_epoch': 4.875674798375084}

In [45]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [46]:
#Takes 1m40
y_pred = model.predict(test_ds)

531/531 ━━━━━━━━━━━━━━━━━━━━ 73s 128ms/step


In [47]:
from sklearn import metrics

In [48]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [49]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1180,1280,1281,1300,1301,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,,,
10,351,11,0,0,2,7,0,8,4,0,0,0,2,0,2,3,1,0,93,115,1,10,0,1,0,12,0
40,19,222,19,2,2,34,0,16,9,14,0,1,0,5,2,0,6,0,48,31,28,4,1,0,34,1,4
50,1,5,117,10,9,7,1,24,2,23,1,0,14,6,1,1,12,0,1,11,17,19,2,2,49,0,1
60,1,6,16,110,0,3,0,4,2,11,0,0,0,0,0,0,0,0,1,2,7,3,0,0,0,0,0
1140,6,8,4,0,230,18,16,153,7,7,0,0,12,0,6,1,7,0,9,17,10,5,1,4,11,0,2
1160,3,3,0,0,0,731,0,3,2,0,0,0,0,0,1,0,0,0,14,26,6,1,0,0,1,0,0
1180,7,2,3,0,6,12,31,17,6,2,1,1,9,1,1,0,8,0,11,17,6,0,2,2,5,1,2
1280,2,3,15,2,25,7,3,522,30,116,2,10,21,24,9,3,52,12,2,17,6,12,5,9,63,1,1
1281,7,6,3,1,3,32,2,103,98,3,1,6,5,5,0,6,22,1,8,41,5,20,4,6,14,4,8


In [50]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.606642480204894, 0.6252842579596326)

In [51]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [52]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

,precision,recall,f1-score,support
10,0.541667,0.563403,0.552321,623.000000
40,0.683077,0.442231,0.536880,502.000000
50,0.451737,0.348214,0.393277,336.000000
60,0.833333,0.662651,0.738255,166.000000
1140,0.771812,0.430712,0.552885,534.000000
1160,0.799781,0.924147,0.857478,791.000000
1180,0.553571,0.202614,0.296651,153.000000
1280,0.384106,0.535934,0.447492,974.000000
1281,0.471154,0.236715,0.315113,414.000000
1300,0.732990,0.704658,0.718545,1009.000000


In [53]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.611800,0.554503,0.562842,629.037037
std,0.173530,0.204765,0.173717,420.759339
min,0.250000,0.154618,0.228148,153.000000
25%,0.480396,0.417219,0.420385,310.000000
50%,0.560954,0.567568,0.552321,534.000000
75%,0.748114,0.687733,0.704786,953.500000
max,0.913891,0.936782,0.857478,2042.000000


In [54]:
# positive correlation between support and another measure can suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.460940,0.783567,0.039116
recall,0.460940,1.000000,0.900499,0.067109
f1-score,0.783567,0.900499,1.000000,0.067116
support,0.039116,0.067109,0.067116,1.000000


In [55]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.606642480204894

In [56]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 7,
 'minutes_per_epoch': 5.895813247892591,
 'weighted_avg_f1_score': 0.606642480204894,
 'min_f1_score': np.float64(0.22814814814814816),
 'std_f1_score': np.float64(0.17371693227248358)}

## Update tracker

In [57]:
tracker['comment']="Set augment=False. The train curve is lower, suggesting data augmentation increases overfitting! Validation results are also much better."
tracker['comment']

'Set augment=False. The train curve is lower, suggesting data augmentation increases overfitting! Validation results are also much better.'

In [ ]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmax(model_history.history['val_accuracy'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_accuracy = model_history.history['val_accuracy'][best_epoch_in_session_idx]
    tracker['val_accuracy'] = best_val_accuracy

    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_arch-{arch_version}_epoch_index-{best_epoch_global:02d}_val_accuracy-{best_val_accuracy:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_path)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print(f"Effacement de l'ancien modèle chargé {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le modèle chargé.")
    keep_candidate=False


Le candidat n'a pas battu le modèle chargé.


In [ ]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1

NameError: name 'best_epoch_global' is not defined

In [ ]:
to_track=['arch_version','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate','timestamp','modality']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model, base_model)

In [ ]:
tracker

{'X_train.shape[0]': 67932,
 'actual_epochs': 3,
 'minutes_per_epoch': 5.895813247892591,
 'weighted_avg_f1_score': 0.6252842579596326,
 'min_f1_score': np.float64(0.25906735751295334),
 'std_f1_score': np.float64(0.18303657733753811),
 'comment': 'Set augment=False. The train curve is lower, suggesting data augmentation increases overfitting! Validation results are also much better.',
 'val_accuracy': 0.6365402936935425,
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-01_val_accuracy-0.6365_f1-0.6253.keras',
 'epoch_index': np.int64(1),
 'total_epochs': np.int64(2),
 'subversion': 9,
 'max_epochs': 3,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'timestamp': '20251006-141509',
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [ ]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [ ]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [ ]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [ ]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, loaded_model=load_model, log_file_path=log_file_path)

Log pour l'expérience subversion 9 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,arch_version,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,timestamp,comment,best_model_path,modality
0,2,False,6793,32,1.998601,10,0.001,0.569713,0.528593,0.000000,0.247884,256_128_64_32,16_16,None,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras,image
1,2,False,20380,32,3.867331,8,0.001,0.584609,0.563841,0.000000,0.239801,256_128_64_32,16_16,None,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras,image
2,3,False,6793,32,2.024143,10,0.001,0.566710,0.527659,0.000000,0.243439,128_64_32_16,8_8,None,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which suggests overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras,image
3,4,True,6793,32,1.954997,13,0.001,0.541922,0.490818,0.000000,0.270646,128_64_32_16,8_8,None,"Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.",artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras,image
4,4,True,20380,32,3.115025,4,0.001,0.562529,0.499731,0.000000,0.276383,128_64_32_16,8_8,None,Increased frac from .1 to .3.,artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-03_val_loss-1.7963_f1-0.4997.keras,image
5,5,True,20380,32,3.080263,3,0.001,0.566945,0.506920,0.000000,0.255305,128_64_32_16,8_8,None,Removed dropout layer before softmax. Performance better but more overfitting.,artifacts/on_images/deep_learning/v1/best_model_sv-5_epoch_index-02_val_loss-1.7414_f1-0.5069.keras,image
6,6,True,6793,32,1.781595,6,0.001,0.558585,0.537182,0.000000,0.223538,256_128_64_32,16_16,None,Frac back to .1. Reverted layers/embeddings to higher sizes and uncommented Dropout before softmax.,artifacts/on_images/deep_learning/v1/best_model_sv-6_epoch_index-05_val_loss-1.9153_f1-0.5372.keras,image
7,7,True,6793,16,2.911669,5,0.001,0.473976,0.410351,0.000000,0.261518,256_128_64_32,16_16,20251004-151611,Unfreezed base model. Batch size from 32 to 16 because memory error.,artifacts/on_images/deep_learning/v1/best_model_sv-7_epoch_index-04_val_loss-2.1782_f1-0.4104.keras,image
8,8,True,6793,32,1.819877,4,0.001,0.573775,0.523869,0.000000,0.235614,256_128_64_32,16_16,20251006-101001,"Unfreezed base model, so batch size back to 32. Lowered first dropout rate from .5 to .2.",artifacts/on_images/deep_learning/v1/best_model_sv-8_epoch_index-03_val_loss-1.9175_f1-0.5239.keras,image
9,9,True,67932,32,8.565354,3,0.001,0.591733,0.570463,0.012821,0.211199,256_128_64_32,16_16,20251006-130338,Full training set. Set callbacks to val_accuracy instead of val_loss.,artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-01_val_accuracy-0.5917_f1-0.5705.keras,image


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
